# Is our run actually deterministic? — an empirical test

Walking the pipeline stage by stage — **prep → minimize → dynamics** — and measuring, on *your* hardware, exactly what is and isn't bit-reproducible. Run this first if you care about reproducing a canonical run.

Reproducibility is the baseline you should expect of good MD, not a feature — but on modern GPUs getting it *bit-for-bit* takes deliberate care (fresh contexts, deterministic forces, seeded preparation). This notebook shows where that care is needed and what happens without it; the caveats it surfaces are why we archive the prepared system, not just the seed.

In [ ]:
# --- environment on-ramp: make sure the MD stack + the module are importable in THIS kernel ---
import importlib.util, sys, os, subprocess
_missing = [m for m in ("openmm", "pdbfixer", "mdtraj") if importlib.util.find_spec(m) is None]
if _missing and "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openmm", "pdbfixer", "mdtraj"], check=False)
    _missing = [m for m in _missing if importlib.util.find_spec(m) is None]
if _missing:
    raise SystemExit(f"Missing in this kernel: {_missing}. Select your MD-tutorial conda kernel "
                     "(Kernel > Change Kernel). If it isn't built yet: conda env create -f environment.yml; "
                     "if it exists but is stale: conda env update -f environment.yml.")
if importlib.util.find_spec("mdtutorial") is None and not os.path.exists("mdtutorial.py"):
    import urllib.request
    URL = os.environ.get("MDTUTORIAL_URL", "https://raw.githubusercontent.com/todd471/MD_tutorial/main/mdtutorial.py")
    try: urllib.request.urlretrieve(URL, "mdtutorial.py"); print("fetched mdtutorial.py")
    except Exception as e: print("Place mdtutorial.py next to this notebook.", e)
import numpy as np, matplotlib.pyplot as plt
import openmm as mm
from openmm import app, unit
import mdtraj as md
import mdtutorial as mdt

In [ ]:
PDB_ID = "1L2Y"
TEMP   = 300 * unit.kelvin
SEED   = 2024
OUT    = "determinism_out"
K      = int(os.environ.get("K", "3"))          # identical repeats to compare
DET_PS = int(os.environ.get("DET_PS", "50"))    # production length per repeat (ps) -- enough for chaos to show

### Build the system (shared core) and minimize once
Uses `mdtutorial.prepare_system` — the same seeded, Reference-platform prep the tutorial uses. The hardware report tells you which platform you're on, which sets what reproducibility to expect.

In [ ]:
prep = mdt.prepare_system(PDB_ID, seed=SEED, out_root=OUT, verbose=False)
mdt.print_hardware_report(prep.hardware)
print("\nminimized; ready for the three experiments.")

### Experiment 1 — is the from-scratch PREP deterministic across separate processes?
The honest test is *cross-process* (a fresh interpreter, like an independent reproduction). We prep twice per mode and compare atom positions: the **default** H fix-up platform vs the **Reference** platform (which is what `mdtutorial.repair` uses).

In [ ]:
import textwrap
_PROBE = textwrap.dedent("""
    import sys, random, numpy as np, openmm as mm
    from openmm import app, unit
    from pdbfixer import PDBFixer
    fx = PDBFixer("%s"); fx.findMissingResidues(); fx.findNonstandardResidues()
    fx.removeHeterogens(keepWater=False); fx.findMissingAtoms(); fx.addMissingAtoms()
    ff = app.ForceField("charmm36.xml","charmm36/water.xml")
    m = app.Modeller(fx.topology, fx.positions)
    m.delete([a for a in m.topology.atoms() if a.element==app.element.hydrogen])
    random.seed(2024); np.random.seed(2024)
    plat = mm.Platform.getPlatformByName("Reference") if sys.argv[2]=="ref" else None
    m.addHydrogens(ff, pH=7.0, platform=plat)
    random.seed(2024); np.random.seed(2024)
    m.addSolvent(ff, model="tip3p", padding=1.0*unit.nanometer, neutralize=True)
    np.save(sys.argv[1], np.array(m.positions.value_in_unit(unit.nanometer)))
""") % mdt.outp(f"{PDB_ID}.pdb", OUT)
open("_prep_probe.py", "w").write(_PROBE)
def prep_twice(mode):
    subprocess.run([sys.executable, "_prep_probe.py", "pa.npy", mode], check=True)
    subprocess.run([sys.executable, "_prep_probe.py", "pb.npy", mode], check=True)
    a, b = np.load("pa.npy"), np.load("pb.npy")
    return float(np.abs(a - b).max()) if a.shape == b.shape else float("nan")
d_def, d_ref = prep_twice("default"), prep_twice("ref")
print("from-scratch prep, TWO SEPARATE PROCESSES (cross-process = the honest test):")
print(f"  default platform for the H fix-up : max|Δx| = {d_def*10:.3e} A  ->  {'DIFFERS (non-deterministic)' if d_def>0 else 'identical'}")
print(f"  Reference platform (THE FIX)      : max|Δx| = {d_ref*10:.3e} A  ->  {'DIFFERS' if d_ref>0 else 'IDENTICAL -- prep is deterministic'}")
for _f in ("_prep_probe.py", "pa.npy", "pb.npy"):
    if os.path.exists(_f): os.remove(_f)

### Experiment 2 — does MINIMIZATION reproduce?
Re-minimize the *same* solvated input K times and compare the minimized coordinates. Deterministic only with CUDA + `DeterministicForces`; on OpenCL/CPU it did not reproduce bit-for-bit in our tests (we measure this, we don't assert why).

In [ ]:
mins = []
for k in range(K):
    integk = mm.LangevinMiddleIntegrator(TEMP, 1 / unit.picosecond, 0.002 * unit.picoseconds)
    sk = app.Simulation(prep.topology, prep.system, integk, prep.platform, prep.plat_props)
    sk.context.setPositions(prep.modeller.positions)          # the pre-minimization solvated positions
    sk.minimizeEnergy()
    mins.append(sk.context.getState(getPositions=True).getPositions(asNumpy=True).value_in_unit(unit.nanometer))
for k in range(1, K):
    dmax = float(np.abs(mins[k] - mins[0]).max()) * 10
    print(f"minimization {k} vs 0:  max |Δx| = {dmax:.3e} Å  ->  {'IDENTICAL' if dmax == 0.0 else 'DIFFERS'}")

### Experiment 3 — do the DYNAMICS reproduce (from a fixed start)?
Run the same seed K times, **each in a fresh Context** (a fresh Context per run reproduces the seeded stream for us; re-using one did not). This reproduced bit-for-bit for us even on OpenCL (the dynamics; the cause we leave unasserted).

In [ ]:
import gc
trajs = []
for k in range(K):
    integ_k = mm.LangevinMiddleIntegrator(TEMP, 1 / unit.picosecond, 0.002 * unit.picoseconds)
    integ_k.setRandomNumberSeed(SEED)                         # seed applied at Context creation
    sim_k = app.Simulation(prep.topology, prep.system, integ_k, prep.platform, prep.plat_props)
    sim_k.context.setPositions(prep.min_positions)
    sim_k.context.setVelocitiesToTemperature(TEMP, SEED)      # identical initial velocities every repeat
    dcd = mdt.outp(f"det_run{k}.dcd", OUT)
    rep = app.DCDReporter(dcd, 100); sim_k.reporters.append(rep)   # frame every 0.2 ps
    sim_k.step(DET_PS * 500); sim_k.reporters.clear()
    try:
        rep._out.flush(); os.fsync(rep._out.fileno()); rep._out.close()
    except Exception:
        pass
    rep = None; gc.collect(); trajs.append(dcd); print(f"repeat {k} done -> {dcd}")

mdtop = md.Topology.from_openmm(prep.topology)                 # in-memory topology that produced these DCDs
ref = md.load(trajs[0], top=mdtop)
_dev = prep.hardware.get("DeviceName", prep.platform.getName())
print(f"\nplatform: {prep.platform.getName()}  |  device: {_dev}  |  {ref.n_frames} frames x {ref.n_atoms} atoms")
devs = []
for k in range(1, K):
    t = md.load(trajs[k], top=mdtop)
    dmax = float(np.abs(t.xyz - ref.xyz).max()) * 10
    dev = np.sqrt(((t.xyz - ref.xyz) ** 2).sum(-1).mean(-1)) * 10
    devs.append(dev)
    print(f"repeat {k} vs 0:  max |Δx| = {dmax:.3e} Å  ->  {'BITWISE IDENTICAL' if dmax == 0.0 else 'DIVERGES'}")
tt = np.arange(ref.n_frames) * 0.2
anydiff = any(d.max() > 0 for d in devs)
plt.figure(figsize=(6.8, 3.9))
for k, dev in enumerate(devs, start=1):
    plt.plot(tt, dev, lw=1.2, label=f"repeat {k} vs 0")
plt.yscale("log" if anydiff else "linear")
plt.xlabel("time (ps)"); plt.ylabel("coord deviation from repeat 0 (Å)")
plt.title(f"Dynamics determinism on {_dev}: " + ("trajectories DIVERGE (chaos)" if anydiff else "bit-identical (flat 0)"))
plt.legend(); plt.tight_layout(); plt.show()

### Reading the result
- **Exp 1 (prep):** the Reference-platform fix-up makes prep identical across processes; the default platform may differ. `mdtutorial.repair` bakes in the fix.
- **Exp 2 (minimization):** bit-identical only on CUDA + `DeterministicForces`; OpenCL/CPU differ. Ship the *minimized* state and this stops mattering.
- **Exp 3 (dynamics):** bit-identical from a fixed start with a fresh Context, even on OpenCL — but still per-GPU-model. Together: the whole pipeline reproduces from a seed on a given GPU model.